# PopulationBucketObservation 查看器

将 `PopulationBucketObservation` 的 `exposed_counts`（形状 `[169, 3]`）还原为手牌字符串（如 `AA`、`AKs`），并以表格形式展示 F/C/R 暴露计数及暴露率。

In [2]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from bayes_poker.strategy.range.mappings import RANGE_169_ORDER

In [3]:
def view_population_bucket_observation(obs_repr: str) -> pd.DataFrame:
    """解析 PopulationBucketObservation 字符串，还原为手牌字符串表格。

    Args:
        obs_repr: 直接从调试器复制的 PopulationBucketObservation.__repr__ 字符串。

    Returns:
        DataFrame，行为手牌（AA/AKs 等），列为 fold_exposed/call_exposed/raise_exposed
        及各自占对应动作总量的暴露率。
    """
    import re

    # 提取 action_totals（第一个 array(...)）
    totals_match = re.search(
        r"action_totals=array\(\[([^\]]+)\]", obs_repr, re.DOTALL
    )
    if not totals_match:
        raise ValueError("未找到 action_totals")
    totals_str = totals_match.group(1)
    action_totals = np.array(
        [float(x) for x in re.findall(r"[\d.e+\-]+", totals_str)],
        dtype=np.float32,
    )

    # 提取 exposed_counts（后续的 array([[...]])）
    counts_match = re.search(
        r"exposed_counts=array\(\[(.+?)\],\s*dtype=float32\)",
        obs_repr,
        re.DOTALL,
    )
    if not counts_match:
        raise ValueError("未找到 exposed_counts")
    counts_str = counts_match.group(1)

    # 逐行解析 [a, b, c] 三元组
    rows = re.findall(r"\[([^\[\]]+)\]", counts_str)
    if len(rows) != 169:
        raise ValueError(f"期望 169 行，实际 {len(rows)} 行")

    exposed = np.array(
        [[float(v) for v in re.findall(r"[\d.e+\-]+", row)] for row in rows],
        dtype=np.float32,
    )  # shape [169, 3]

    # 构建 DataFrame
    df = pd.DataFrame(
        {
            "hand": list(RANGE_169_ORDER),
            "fold_exposed": exposed[:, 0],
            "call_exposed": exposed[:, 1],
            "raise_exposed": exposed[:, 2],
        }
    )

    # 计算暴露率（除以对应动作的总数）
    for i, col_name in enumerate(["fold_rate", "call_rate", "raise_rate"]):
        total = action_totals[i]
        src_col = ["fold_exposed", "call_exposed", "raise_exposed"][i]
        df[col_name] = (df[src_col] / total * 100).round(4) if total > 0 else 0.0

    return df


def show_obs(obs_repr: str, min_raise_exposed: float = 0.0) -> pd.DataFrame:
    """展示手牌暴露数据，可按最小加注暴露量过滤。

    Args:
        obs_repr: PopulationBucketObservation.__repr__ 字符串。
        min_raise_exposed: 过滤条件，仅保留 raise_exposed >= 该值的手牌。

    Returns:
        过滤后的 DataFrame。
    """
    df = view_population_bucket_observation(obs_repr)
    if min_raise_exposed > 0:
        df = df[df["raise_exposed"] >= min_raise_exposed].copy()
    return df.sort_values("raise_exposed", ascending=False).reset_index(drop=True)


In [ ]:
# ============================================================
# 把你的 PopulationBucketObservation repr 字符串粘贴到这里
# ============================================================
obs_repr = """"""

# 展示全部
df_all = view_population_bucket_observation(obs_repr)
pd.set_option("display.max_rows", 169)
df_all


In [7]:
# 只看 raise_exposed >= 500 的手牌，按加注暴露量排序
show_obs(obs_repr, min_raise_exposed=500)


,hand,fold_exposed,call_exposed,raise_exposed,fold_rate,call_rate,raise_rate
0,AKo,0.0,521.0,14721.0,0.0,0.4959,1.0636
1,AQo,0.0,508.0,12385.0,0.0,0.4835,0.8948
2,AJo,0.0,607.0,10251.0,0.0,0.5777,0.7406
3,QQ,0.0,319.0,8967.0,0.0,0.3036,0.6479
4,KK,0.0,405.0,8959.0,0.0,0.3855,0.6473
5,JJ,0.0,328.0,8792.0,0.0,0.3122,0.6352
6,KQo,0.0,601.0,8645.0,0.0,0.5720,0.6246
7,TT,0.0,318.0,8328.0,0.0,0.3027,0.6017
8,AA,0.0,478.0,8304.0,0.0,0.4550,0.6000
9,ATo,0.0,613.0,7191.0,0.0,0.5834,0.5196
